In [ ]:
# Código para resolver o problema de tomografia quântica para N q-bits.
import numpy as np
import scipy.optimize as scpo
from scipy.linalg import sqrtm
import matplotlib.pyplot as plt
import cvxpy as cp
import itertools   
from itertools import product
import time  
from tqdm import tqdm  
import contextlib  
import io
from functools import partial
class QuantState:
    def __init__(self, estado):
        """
        Aceita um vetor 1D (estado puro) ou uma matriz 2D (matriz densidade / estado misto).
        """
        estado = np.array(estado, dtype=complex)

        if estado.ndim == 1:
            norma = np.linalg.norm(estado)
            psi = estado / norma
            self.rho = np.outer(psi, psi.conj())

        elif estado.ndim == 2:
            if estado.shape[0] != estado.shape[1]:
                raise ValueError("A matriz densidade deve ser quadrada.")
            traco = np.trace(estado)
            self.rho = estado / traco

        # Preciso definir condições para dimensões maiores
        else:
            raise ValueError("O estado deve ser um vetor 1D (|psi>) ou matriz 2D (rho).")

    def measure(self, obs):
        """
        Mede o estado em relação a um observável (matriz hermitiana).
        Retorna o valor esperado <obs> = Tr(rho * obs).
        """
        return np.trace(self.rho @ obs).real
def parametros_para_T_geral(params, n_qubits):
    """Constrói a matriz triangular inferior T para N qubits (D = 2^N)."""
    D = 2**n_qubits
    li, lj = np.tril_indices(D, k=-1)          # índices abaixo da diagonal
    n_off = len(li)                            # D(D-1)/2 elementos complexos

    T = np.zeros((D, D), dtype=complex)
    T[np.arange(D), np.arange(D)] = params[:D]
    T[li, lj] = params[D:D + n_off] + 1j * params[D + n_off:]
    return T


def parametros_para_rho_geral(params, n_qubits):
    """Gera rho para qualquer número de qubits."""
    T = parametros_para_T_geral(params, n_qubits)
    T_dag_T = T.conj().T @ T
    traco = np.trace(T_dag_T).real             #  .real (o traço de T†T é real)

    if np.isclose(traco, 0):
        D = 2**n_qubits
        return np.eye(D, dtype=complex) / D

    return T_dag_T / traco

def funcao_de_custo_geral(params, mediadores_T, dados_exp, n_qubits):
    rho_candidato = parametros_para_rho_geral(params, n_qubits)
    previsoes = np.einsum('ij,mji->m', rho_candidato, mediadores_T).real
    return previsoes - dados_exp
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)


# Gera as 4**n - 1 strings de Pauli (todas menos I⊗...⊗I) via produto de Kronecker.
def gerar_paulis(n_qubits):
    I = np.eye(2, dtype=complex)
    base = [I, sigma_x, sigma_y, sigma_z]
    saida = []
    for combo in itertools.product(base, repeat=n_qubits):
        P = combo[0]
        for c in combo[1:]:
            P = np.kron(P, c)
        saida.append(P)
    return saida[1:]        # remove a identidade (o valor esperado dela é sempre 1)


# Fidelidade entre duas matrizes densidade: F = (Tr sqrt( sqrt(A) B sqrt(A) ))^2
def sqrt_psd(A):
    w, V = np.linalg.eigh((A + A.conj().T) / 2)
    return (V * np.sqrt(np.clip(w, 0, None))) @ V.conj().T

def fidelidade(rho_a, rho_b):
    sa = sqrt_psd(rho_a)
    return np.real(np.trace(sqrt_psd(sa @ rho_b @ sa)))**2
# %% CELULA 1 -- Jacobiano analitico
def jacobiano_analitico(params, mediadores_Q, dados_exp, n_qubits):
    """
    Jacobiano analitico de funcao_de_custo_geral em relacao aos parametros reais.
 
    Deducao (vale reproduzir no relatorio):
      rho(T) = T^dagger T / s,  s = Tr(T^dagger T)
      previsao_m(T) = Tr(rho @ Q_m) = f_m(T)/s,  f_m = Tr(T^dagger T Q_m)
      T e LINEAR em cada parametro real p_i (T = soma_i p_i * B_i, com B_i
      uma matriz-base fixa: E_ii para os p_i diagonais, E_kl para a parte
      real fora da diagonal, i*E_kl para a parte imaginaria). Daqui:
        d f_m/dp_i = 2*Re Tr(T^dagger B_i Q_m)
        d s  /dp_i = 2*Re Tr(T^dagger B_i)
        d previsao_m/dp_i = (2/s) * [Re Tr(T^dagger B_i Q_m) - previsao_m * Re Tr(T^dagger B_i)]
      Usando Tr(T^dagger E_kl Q_m) = (Q_m @ T^dagger)[l,k] e Tr(T^dagger E_kl) = T^dagger[l,k],
      isso fica totalmente vetorizavel (sem laco sobre os parametros).
 
    Validado contra diferenca finita central (erro maximo ~1e-10) para
    n_qubits = 1, 2, 3.
    """
    D = 2 ** n_qubits
    li, lj = np.tril_indices(D, k=-1)
    n_off = len(li)
 
    T = parametros_para_T_geral(params, n_qubits)
    T_dag = T.conj().T
    s = np.trace(T_dag @ T).real
 
    Q = np.asarray(mediadores_Q)
    previsoes = np.einsum('ij,mji->m', T_dag @ T / s, Q).real
 
    M_all = np.einsum('mij,jk->mik', Q, T_dag)   # M_all[m] = Q_m @ T^dagger
 
    tr_diag = np.diagonal(M_all, axis1=1, axis2=2)
    tr_re = M_all[:, lj, li]
    tr_im = 1j * M_all[:, lj, li]
 
    g_diag = np.diagonal(T_dag)
    g_re = T_dag[lj, li]
    g_im = 1j * T_dag[lj, li]
 
    jac_diag = (2.0 / s) * (tr_diag.real - previsoes[:, None] * g_diag.real[None, :])
    jac_re = (2.0 / s) * (tr_re.real - previsoes[:, None] * g_re.real[None, :])
    jac_im = (2.0 / s) * (tr_im.real - previsoes[:, None] * g_im.real[None, :])
 
    return np.hstack([jac_diag, jac_re, jac_im])

def least_squares(fun, x0, **kwargs):
    # Dicionário com os seus parâmetros otimizados
    parametros_padrao = {
        'method': 'dogbox',
        'x_scale': 'jac',
        'tr_solver': 'lsmr',
        'jac': jacobiano_analitico 
    }
    
    # Atualiza os parâmetros padrão com qualquer outro que você passe na hora
    parametros_padrao.update(kwargs)
    
    # Chama a função original desempaquetando o dicionário
    return scpo.least_squares(fun, x0, **parametros_padrao)


def medir_com_ruido(q_exato, N_shots, rng):
    q_ruido = np.empty_like(q_exato)
    for i, qi in enumerate(q_exato):
        p_mais = (1 + qi) / 2
        outcomes = rng.binomial(N_shots, p_mais)
        q_ruido[i] = 2 * outcomes / N_shots - 1
    return q_ruido
def roda_com_historico(params0, mediadores_Q, dados_exp, n_qubits, **kwargs_least_squares):
    """
    Roda least_squares com verbose=2 e extrai, da propria saida do scipy, o
    custo reportado a cada ITERACAO aceita (nao a cada avaliacao de funcao --
    isso importa quando o Jacobiano e numerico, porque cada iteracao pode
    embutir varias avaliacoes so para estimar o Jacobiano).
    Retorna (resultado, custos) com custos = array 1D, um valor por iteracao.
    """
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        resultado = least_squares(
            funcao_de_custo_geral,
            x0=params0,
            args=(mediadores_Q, dados_exp, n_qubits),
            verbose=2,
            **kwargs_least_squares,
        )
    custos = []
    for linha in buf.getvalue().splitlines():
        campos = linha.split()
        if campos and campos[0].lstrip("-").isdigit():
            custos.append(float(campos[2]))
    return resultado, np.array(custos)


SEED_ESTADO = 123
SEED_CHUTE = 456
N_QUBITS_CONTROLE = 3
 
 
def monta_problema_controle(n_qubits=N_QUBITS_CONTROLE):
    rng_estado = np.random.default_rng(SEED_ESTADO)
    rng_chute = np.random.default_rng(SEED_CHUTE)
    D = 2 ** n_qubits
    mediadores_Q = gerar_paulis(n_qubits)
    estado_verdadeiro = QuantState(rng_estado.normal(size=D) + 1j * rng_estado.normal(size=D))
    q_medidos = np.array([estado_verdadeiro.measure(Q) for Q in mediadores_Q])
    chute_inicial = rng_chute.normal(size=D ** 2)
    return dict(n_qubits=n_qubits, mediadores_Q=mediadores_Q, q_medidos=q_medidos,
                chute_inicial=chute_inicial, estado_verdadeiro=estado_verdadeiro)

In [ ]:
def experimento_jac():
    p = monta_problema_controle()
    resultados = {}
    variantes = {
        "2-point (default)": dict(jac="2-point"),
        "3-point": dict(jac="3-point"),
        "analitico": dict(jac=lambda params, *args: jacobiano_analitico(params, *args)),
    }
    for nome, kwargs in variantes.items():
        t0 = time.time()
        try:
            resultado, custos = roda_com_historico(
                p["chute_inicial"], p["mediadores_Q"], p["q_medidos"], p["n_qubits"], **kwargs)
            dt = time.time() - t0
            rho_rec = parametros_para_rho_geral(resultado.x, p["n_qubits"])
            fid = fidelidade(rho_rec, p["estado_verdadeiro"].rho)
            resultados[nome] = dict(custo_final=resultado.cost, nfev=resultado.nfev,
                                     njev=resultado.njev, fidelidade=fid, tempo=dt, custos=custos)
            print(f"{nome:20s} | custo={resultado.cost:.3e} | nfev={resultado.nfev:4d} | "
                  f"njev={resultado.njev:4d} | fid={fid:.6f} | t={dt*1000:.1f}ms | status={resultado.status}")
        except Exception as e:
            print(f"{nome:20s} | FALHOU: {e}")
    plt.figure(figsize=(7, 5))
    for nome, r in resultados.items():
        if len(r["custos"]) > 0:
            plt.plot(r["custos"], marker="o", label=nome)
    plt.yscale("log"); plt.xlabel("Iteração"); plt.ylabel("Custo")
    plt.title("Convergência para diferentes formas de Jacobiano")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig("exp1_jac.png", dpi=130); plt.close()
    return resultados

In [ ]:
def experimento_method():
    p = monta_problema_controle()
    D = 2 ** p["n_qubits"]; M = len(p["mediadores_Q"]); N = D ** 2
    print(f"M={M} observáveis | N={N} parâmetros reais")
    if M < N:
        print("M < N: 'lm' será pulado (exige nº de resíduos >= nº de variáveis). Isso é "
              "estrutural em tomografia completa de Pauli (M=4^n-1 < N=4^n sempre), por causa "
              "de uma direção de gauge (fase global de T) que nenhuma medida restringe.")
    resultados = {}
    for metodo in ["trf", "lm", "dogbox"]:
        if metodo == "lm" and M < N:
            print(f"{metodo:10s} | pulado"); continue
        resultado, custos = roda_com_historico(
            p["chute_inicial"], p["mediadores_Q"], p["q_medidos"], p["n_qubits"], method=metodo)
        rho_rec = parametros_para_rho_geral(resultado.x, p["n_qubits"])
        fid = fidelidade(rho_rec, p["estado_verdadeiro"].rho)
        resultados[metodo] = dict(custo_final=resultado.cost, nfev=resultado.nfev, custos=custos)
        print(f"{metodo:10s} | custo={resultado.cost:.3e} | nfev={resultado.nfev:4d} | fid={fid:.6f}")
    plt.figure(figsize=(7, 5))
    for metodo, r in resultados.items():
        if len(r["custos"]) > 0:
            plt.plot(r["custos"], marker="o", label=metodo)
    plt.yscale("log"); plt.xlabel("Iteração"); plt.ylabel("Custo")
    plt.title("Convergência: 'trf' vs 'lm' vs 'dogbox'")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig("exp2_method.png", dpi=130); plt.close()
    return resultados

In [ ]:
def _roda_variantes(p, titulo, variantes, arquivo_saida):
    resultados = {}
    for nome, kwargs in variantes.items():
        t0 = time.time()
        resultado, custos = roda_com_historico(
            p["chute_inicial"], p["mediadores_Q"], p["q_medidos"], p["n_qubits"], **kwargs)
        dt = time.time() - t0
        rho_rec = parametros_para_rho_geral(resultado.x, p["n_qubits"])
        fid = fidelidade(rho_rec, p["estado_verdadeiro"].rho)
        resultados[nome] = dict(custo_final=resultado.cost, nfev=resultado.nfev,
                                 fidelidade=fid, tempo=dt, custos=custos)
        print(f"{nome:28s} | custo={resultado.cost:.3e} | nfev={resultado.nfev:4d} | "
              f"fid={fid:.6f} | t={dt*1000:.1f}ms")
    plt.figure(figsize=(7, 5))
    for nome, r in resultados.items():
        if len(r["custos"]) > 0:
            plt.plot(r["custos"], marker="o", label=nome)
    plt.yscale("log"); plt.xlabel("Iteração"); plt.ylabel("Custo"); plt.title(titulo)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(arquivo_saida, dpi=130); plt.close()
    return resultados
 
 
def experimento_x_scale():
    p = monta_problema_controle()
    variantes = {"x_scale=1.0 (default)": dict(x_scale=1.0), "x_scale='jac'": dict(x_scale="jac")}
    return _roda_variantes(p, "Convergência: x_scale", variantes, "exp4_x_scale.png")
 
 
def experimento_tr_solver():
    p = monta_problema_controle()
    variantes = {"tr_solver='exact' (default)": dict(tr_solver="exact"),
                 "tr_solver='lsmr'": dict(tr_solver="lsmr")}
    return _roda_variantes(p, "Convergência: tr_solver", variantes, "exp5_tr_solver.png")
 
 
def experimento_diff_step():
    p = monta_problema_controle()
    variantes = {"diff_step=None (automático)": dict(diff_step=None),
                 "diff_step=1e-3 (grande)": dict(diff_step=1e-3),
                 "diff_step=1e-8 (pequeno demais)": dict(diff_step=1e-8)}
    return _roda_variantes(p, "Convergência: diff_step", variantes, "exp6_diff_step.png")

In [ ]:
 
def experimento_loss(N_shots=20, n_reps=15):
    p = monta_problema_controle()
    q_exato = p["q_medidos"]
    losses = ["linear", "soft_l1", "huber", "cauchy"]
    fids_por_loss = {loss: [] for loss in losses}
    custo_exemplo = {}
    rng = np.random.default_rng(2024)
    for rep in range(n_reps):
        q_ruido = medir_com_ruido(q_exato, N_shots, rng)
        for loss in losses:
            resultado, custos = roda_com_historico(
                p["chute_inicial"], p["mediadores_Q"], q_ruido, p["n_qubits"], loss=loss)
            rho_rec = parametros_para_rho_geral(resultado.x, p["n_qubits"])
            fids_por_loss[loss].append(fidelidade(rho_rec, p["estado_verdadeiro"].rho))
            if rep == 0:
                custo_exemplo[loss] = custos
    print(f"N_shots={N_shots}, {n_reps} repetições\n{'loss':10s} | fidelidade média ± desvio")
    for loss in losses:
        fids = np.array(fids_por_loss[loss])
        print(f"{loss:10s} | {fids.mean():.4f} ± {fids.std():.4f}")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.boxplot([fids_por_loss[l] for l in losses], tick_labels=losses)
    ax1.set_ylabel("Fidelidade"); ax1.set_title(f"Fidelidade final por loss (N_shots={N_shots})")
    ax1.grid(alpha=0.3)
    for loss in losses:
        if len(custo_exemplo[loss]) > 0:
            ax2.plot(custo_exemplo[loss], marker="o", label=loss)
    ax2.set_yscale("log"); ax2.set_xlabel("Iteração"); ax2.set_ylabel("Custo")
    ax2.set_title("Convergência (uma repetição de exemplo)")
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("exp3_loss.png", dpi=130); plt.close()
    return fids_por_loss

In [ ]:
'''
print("=== Experimento 1: jac ===")
experimento_jac()
print("\n=== Experimento 2: method ===")
experimento_method()
print("\n=== Experimento 3: loss (shot noise) ===")
experimento_loss()
print("\n=== Experimento 4: x_scale ===")
experimento_x_scale()
print("\n=== Experimento 5: tr_solver ===")
experimento_tr_solver()
print("\n=== Experimento 6: diff_step ===")
experimento_diff_step()
'''